# EFSM Local Full Speech Demo

This notebook launches the final Gradio demo on a local PC.

Real demo path:

```text
microphone audio -> Whisper ASR -> Qwen2.5-7B-Instruct + EFSM LoRA -> TTS audio reply
```

Use mock mode first to confirm that the Gradio UI opens. The real model requires an NVIDIA GPU with CUDA-enabled PyTorch and enough VRAM for 4-bit Qwen2.5-7B inference.

## 0. Local Setup Notes

Before running this notebook locally:

1. Open it from the repository root or make sure the current working directory is `empathetic-voice-llm`.
2. Use a Python environment with CUDA-enabled PyTorch if running the real model.
3. If the Hugging Face adapter repo is private, provide a token when prompted in Cell 1.
4. Keep the token private. Do not save it in GitHub, README files, or committed notebooks.

In [ ]:
# Cell 1 - Optional Hugging Face token
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    token = getpass("Paste HF_TOKEN if needed, otherwise press Enter: ").strip()
    if token:
        os.environ["HF_TOKEN"] = token

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. Confirm Working Directory

This cell checks whether the notebook is running from the project root.

In [ ]:
from pathlib import Path
import os

cwd = Path.cwd()
print("Current directory:", cwd)

if not (cwd / "demo" / "app.py").exists():
    raise FileNotFoundError(
        "demo/app.py was not found. Open this notebook from the repository root "
        "or change the working directory to empathetic-voice-llm."
    )

print("Project root looks correct.")

## 2. Install Python Dependencies

`requirements.txt` does not install PyTorch. If CUDA PyTorch is missing, install the correct build from the official PyTorch website for the local machine's CUDA version.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## 3. GPU Check

The real demo requires CUDA. If CUDA is unavailable, run only mock mode.

In [ ]:
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    print("GPU count:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"GPU {i}: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")
except Exception as exc:
    print("Could not import/check torch:", exc)

## 4. UI Mock Test

Run this first. It launches the interface without loading Whisper or Qwen. Stop the cell before running the real demo.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "demo/app.py", "--mock"], check=True)

## 5. Real Full Demo

Run this after the mock UI works. Open the local URL printed by Gradio, usually:

```text
http://127.0.0.1:7860
```

This loads:

- `Qwen/Qwen2.5-7B-Instruct`
- LoRA adapter `tasbid001/efsm-checkpoints-fixed`, subfolder `checkpoint-2667`
- Whisper ASR
- TTS output

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "demo/app.py",
    "--adapter-subfolder", "checkpoint-2667",
], check=True)

## 6. Lower-Memory Option

If the real demo runs out of VRAM, try shorter responses.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "demo/app.py",
    "--adapter-subfolder", "checkpoint-2667",
    "--max-new-tokens", "100",
], check=True)

## 7. Public Link Option

If the demo needs to be opened from another device, launch Gradio with `--share`.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "demo/app.py",
    "--adapter-subfolder", "checkpoint-2667",
    "--max-new-tokens", "100",
    "--share",
], check=True)